# Routing Pattern

**Pattern**: Decision-making to route tasks to the most appropriate handler based on input.

**Example**: Customer Support Ticket Router

**Workflow**:
```
                         ┌→ [Billing Handler]   ─┐
START → [Classifier] ───┼→ [Technical Handler] ─┼→ [Format Response] → END
                         └→ [General Handler]   ─┘
```

Based on the customer query:
- Billing queries → Billing Handler (refunds, payments, invoices)
- Technical queries → Technical Handler (bugs, errors, how-to)
- General queries → General Handler (everything else)

## Setup

In [ ]:
from typing import TypedDict, Literal

from langgraph.graph import StateGraph, START, END

## Define State

In [ ]:
class TicketState(TypedDict):
    # Input
    customer_name: str
    query: str
    
    # After classification
    category: str           # billing, technical, general
    confidence: float       # 0.0 to 1.0
    
    # After handler
    handler_response: str
    suggested_actions: list[str]
    escalate: bool
    
    # Final output
    formatted_response: str

## Define Classifier Node

This node analyzes the query and determines which handler should process it.

In [ ]:
# Keywords for classification (in production, use an LLM)
BILLING_KEYWORDS = ["refund", "payment", "invoice", "charge", "bill", "subscription", "cancel", "price", "cost", "money"]
TECHNICAL_KEYWORDS = ["error", "bug", "crash", "not working", "broken", "fix", "issue", "problem", "help", "how to", "install", "update"]

def classify_ticket(state: TicketState) -> dict:
    """Classify the ticket into billing, technical, or general."""
    query = state["query"].lower()
    
    # Count keyword matches
    billing_score = sum(1 for kw in BILLING_KEYWORDS if kw in query)
    technical_score = sum(1 for kw in TECHNICAL_KEYWORDS if kw in query)
    
    # Determine category based on scores
    if billing_score > technical_score and billing_score > 0:
        category = "billing"
        confidence = min(billing_score * 0.3, 1.0)
    elif technical_score > billing_score and technical_score > 0:
        category = "technical"
        confidence = min(technical_score * 0.3, 1.0)
    else:
        category = "general"
        confidence = 0.5  # Default confidence for general
    
    return {
        "category": category,
        "confidence": confidence
    }

## Define Handler Nodes

Each handler specializes in a specific type of query.

In [ ]:
def billing_handler(state: TicketState) -> dict:
    """Handle billing-related queries."""
    query = state["query"].lower()
    
    # Determine specific billing action
    if "refund" in query:
        response = "I understand you're requesting a refund. I've initiated the refund process for your account."
        actions = [
            "Refund request created (Ticket #RF-2024)",
            "Expected processing time: 5-7 business days",
            "Confirmation email sent to registered address"
        ]
        escalate = False
    elif "cancel" in query:
        response = "I can help you with cancellation. Before proceeding, I'd like to understand your concerns."
        actions = [
            "Cancellation request noted",
            "Retention offer available: 20% off for 3 months",
            "Schedule callback with retention team"
        ]
        escalate = True  # Escalate to retention team
    elif "invoice" in query or "bill" in query:
        response = "I can help you with your invoice. Your latest invoice has been sent to your email."
        actions = [
            "Invoice #INV-2024-0426 resent to email",
            "Payment due date: May 15, 2026",
            "Download link valid for 7 days"
        ]
        escalate = False
    else:
        response = "I'll connect you with our billing team for detailed assistance."
        actions = [
            "Ticket created for billing team",
            "Priority: Normal",
            "Expected response: 24 hours"
        ]
        escalate = True
    
    return {
        "handler_response": response,
        "suggested_actions": actions,
        "escalate": escalate
    }

In [ ]:
def technical_handler(state: TicketState) -> dict:
    """Handle technical support queries."""
    query = state["query"].lower()
    
    if "error" in query or "crash" in query or "bug" in query:
        response = "I see you're experiencing a technical issue. Let me help troubleshoot."
        actions = [
            "Bug report created (Ticket #BUG-2024)",
            "Try: Clear cache and restart application",
            "If issue persists, engineering team will investigate"
        ]
        escalate = True  # Escalate bugs to engineering
    elif "how to" in query or "help" in query:
        response = "I'd be happy to guide you through this process."
        actions = [
            "Step-by-step guide sent to your email",
            "Video tutorial link: help.example.com/tutorials",
            "Live chat available for real-time assistance"
        ]
        escalate = False
    elif "install" in query or "update" in query:
        response = "I can help you with installation/update process."
        actions = [
            "Latest version: v4.2.1",
            "Download link sent to email",
            "Installation guide attached"
        ]
        escalate = False
    else:
        response = "Our technical team will investigate this issue."
        actions = [
            "Technical ticket created",
            "Priority: High",
            "Expected response: 4 hours"
        ]
        escalate = True
    
    return {
        "handler_response": response,
        "suggested_actions": actions,
        "escalate": escalate
    }

In [ ]:
def general_handler(state: TicketState) -> dict:
    """Handle general queries that don't fit other categories."""
    query = state["query"].lower()
    
    if "thank" in query or "appreciate" in query:
        response = "You're welcome! We're always happy to help."
        actions = [
            "Feedback recorded",
            "No further action needed"
        ]
        escalate = False
    elif "speak" in query or "human" in query or "agent" in query:
        response = "I'll connect you with a human agent right away."
        actions = [
            "Transferring to live agent",
            "Estimated wait time: 2 minutes",
            "Queue position: #3"
        ]
        escalate = True
    else:
        response = "Thank you for reaching out. Let me help you with your inquiry."
        actions = [
            "Query logged for review",
            "FAQ link: help.example.com/faq",
            "Response within 24 hours if needed"
        ]
        escalate = False
    
    return {
        "handler_response": response,
        "suggested_actions": actions,
        "escalate": escalate
    }

## Define Format Response Node

In [ ]:
def format_response(state: TicketState) -> dict:
    """Format the final response to send to customer."""
    
    lines = []
    lines.append("=" * 50)
    lines.append("       CUSTOMER SUPPORT RESPONSE")
    lines.append("=" * 50)
    
    lines.append(f"\nHello {state['customer_name']},")
    lines.append(f"\nYour Query: \"{state['query']}\"")
    
    lines.append(f"\n📋 Category: {state['category'].upper()}")
    lines.append(f"   Confidence: {state['confidence']*100:.0f}%")
    
    lines.append("\n" + "-" * 50)
    lines.append("RESPONSE")
    lines.append("-" * 50)
    lines.append(f"\n{state['handler_response']}")
    
    lines.append("\n" + "-" * 50)
    lines.append("ACTIONS TAKEN")
    lines.append("-" * 50)
    for action in state['suggested_actions']:
        lines.append(f"  ✓ {action}")
    
    if state['escalate']:
        lines.append("\n⚠️  ESCALATED: A specialist will follow up shortly.")
    
    lines.append("\n" + "=" * 50)
    lines.append("Thank you for contacting us!")
    lines.append("=" * 50)
    
    return {"formatted_response": "\n".join(lines)}

## Define Routing Function

This function reads the state and returns which node to route to.

**This is the key to conditional edges!**

In [ ]:
def route_ticket(state: TicketState) -> Literal["billing", "technical", "general"]:
    """
    Routing function for conditional edge.
    Returns the name of the next node based on category.
    """
    category = state["category"]
    
    if category == "billing":
        return "billing"
    elif category == "technical":
        return "technical"
    else:
        return "general"

## Build the Graph with Conditional Edges

In [ ]:
# Create graph
graph = StateGraph(TicketState)

# Add nodes
graph.add_node("classifier", classify_ticket)
graph.add_node("billing", billing_handler)
graph.add_node("technical", technical_handler)
graph.add_node("general", general_handler)
graph.add_node("format_response", format_response)

# Add edges
graph.add_edge(START, "classifier")

# CONDITIONAL EDGE: Route based on classification
graph.add_conditional_edges(
    "classifier",           # Source node
    route_ticket,           # Routing function
    {                       # Mapping: routing function return value → node name
        "billing": "billing",
        "technical": "technical",
        "general": "general"
    }
)

# All handlers lead to format_response
graph.add_edge("billing", "format_response")
graph.add_edge("technical", "format_response")
graph.add_edge("general", "format_response")

graph.add_edge("format_response", END)

# Compile
app = graph.compile()

## Visualize the Graph

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## Test Case 1: Billing Query (Refund)

In [ ]:
result = app.invoke({
    "customer_name": "Rahul",
    "query": "I want a refund for my last payment, it was charged twice"
})

print(result["formatted_response"])

## Test Case 2: Technical Query (Error/Bug)

In [ ]:
result = app.invoke({
    "customer_name": "Priya",
    "query": "The app keeps crashing with an error when I try to upload files"
})

print(result["formatted_response"])

## Test Case 3: Technical Query (How-to)

In [ ]:
result = app.invoke({
    "customer_name": "Amit",
    "query": "How to install the latest update on my phone?"
})

print(result["formatted_response"])

## Test Case 4: Billing Query (Cancellation)

In [ ]:
result = app.invoke({
    "customer_name": "Sneha",
    "query": "I want to cancel my subscription, the price is too high"
})

print(result["formatted_response"])

## Test Case 5: General Query (Talk to Human)

In [ ]:
result = app.invoke({
    "customer_name": "Vikram",
    "query": "I want to speak to a human agent please"
})

print(result["formatted_response"])

## Test Case 6: General Query (Unclassified)

In [ ]:
result = app.invoke({
    "customer_name": "Ananya",
    "query": "What are your office hours?"
})

print(result["formatted_response"])

## Stream Execution (See the Route Taken)

In [ ]:
print("Streaming execution...\n")

for step in app.stream({"customer_name": "Test User", "query": "There's a bug in the payment system, I was charged wrong amount"}):
    for node_name, output in step.items():
        print(f"✓ Node: {node_name}")
        if node_name == "classifier":
            print(f"   → Classified as: {output['category']} (confidence: {output['confidence']*100:.0f}%)")
        elif node_name in ["billing", "technical", "general"]:
            print(f"   → Handler response: {output['handler_response'][:50]}...")
            print(f"   → Escalate: {output['escalate']}")

## Interactive: Try Your Own Query

In [ ]:
# Change these values!
my_name = "Your Name"
my_query = "Your query here..."

result = app.invoke({
    "customer_name": my_name,
    "query": my_query
})

print(result["formatted_response"])

## Key Takeaways

1. **Routing Pattern** uses `add_conditional_edges()` to dynamically choose the next node
2. **Routing function** reads state and returns a string key that maps to a node
3. **Classifier node** determines the category, routing function uses that category
4. **All routes converge** to a common node (format_response) before END
5. **Only one handler executes** per invocation — the one selected by the router

### Conditional Edge Syntax:
```python
graph.add_conditional_edges(
    source_node,      # Where the edge starts
    routing_function, # Function that returns a key
    {                 # Map keys to destination nodes
        "key1": "node1",
        "key2": "node2",
    }
)
```